# IM Session 29 — GenAI Foundations & Prompt Engineering

**Batch:** IITP-AIMLTN-2605  |  **Curriculum session:** 12.1  |  **Module 3: GenAI & Agents**

## Learning Objectives

By the end of this notebook you will be able to:

- **LO 1** — Explain how an LLM turns text into tokens, and why the context window is a hard budget
- **LO 2** — Describe probabilistic text generation, the role of temperature, and why hallucinations happen
- **LO 3** — Build and compare zero-shot, few-shot and chain-of-thought prompts, including their token cost

## The situation

You are the analyst for a Tamil Nadu e-commerce support desk. Forty real-shaped support tickets arrive by
email, chat and phone. Someone has to read each one, decide its category and priority, and route it.

Across Module 2 you would have solved this by *training a classifier on labelled data*. In Module 3 you will
solve it by *steering a model somebody else already trained, using words*. This notebook builds the vocabulary
you need to do that well.

> **Runs completely offline.** No API key, no network call, no cost. Every number below is computed from the
> ticket file in the `Datasets` folder. Session 30 covers the real API call.


## Setup

In [1]:
import re
import textwrap
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)          # every random result below is reproducible
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 70)

TICKETS = Path("..") / "Datasets" / "support_tickets.csv"
tickets = pd.read_csv(TICKETS)

print(f"tickets      : {len(tickets)}")
print(f"categories   : {sorted(tickets['category'].unique())}")
print(f"channels     : {tickets['channel'].value_counts().to_dict()}")
tickets.head(3)[["ticket_id", "category", "priority", "ticket_text"]]

tickets      : 40
categories   : ['account', 'billing', 'delivery', 'how_to', 'product_issue', 'refund']
channels     : {'email': 20, 'chat': 15, 'phone': 5}


,ticket_id,category,priority,ticket_text
0,TKT-1001,refund,high,"Ordered a wireless keyboard on 28 July, delivered cracked. I want ..."
1,TKT-1002,delivery,medium,Where is my order? It says shipped since Friday but tracking has n...
2,TKT-1003,billing,high,I was charged twice for the same order. Two debits of INR 4599 on ...


---

## Part 1 — Tokens

An LLM never sees your characters. Before anything happens, your text is cut into **tokens** — the units the
model actually reads and predicts. Tokens are usually *sub-word* pieces: common words are one token, rare or
long words are split into several.

We do not have the real tokenizer library installed here, so we will build a transparent approximation. It
splits on words and punctuation, then breaks long words into ~4-character chunks — which is the same rough
behaviour real byte-pair encoding shows on English text.

> **Be honest about what this is.** This is a *teaching approximation*, not the real thing. It gets you the
> right intuition and a usable estimate; a production token count must come from the provider's own tokenizer.


In [2]:
WORD_RE = re.compile(r"\w+|[^\w\s]")


def tokenize(text, chunk=4):
    """Approximate sub-word tokenization: split on words/punctuation, then chunk long words."""
    out = []
    for piece in WORD_RE.findall(str(text)):
        if len(piece) <= chunk:
            out.append(piece)
        else:
            out.extend(piece[i:i + chunk] for i in range(0, len(piece), chunk))
    return out


def count_tokens(text):
    return len(tokenize(text))


demo = "I was charged twice for the same order. Please reverse one immediately."
print("text   :", demo)
print("tokens :", tokenize(demo))
print("count  :", count_tokens(demo))

text   : I was charged twice for the same order. Please reverse one immediately.
tokens : ['I', 'was', 'char', 'ged', 'twic', 'e', 'for', 'the', 'same', 'orde', 'r', '.', 'Plea', 'se', 'reve', 'rse', 'one', 'imme', 'diat', 'ely', '.']
count  : 21


Notice `immediately` became several pieces while `order` stayed whole. That is the sub-word behaviour that
makes token counts hard to guess by eye — which is exactly why you measure them instead of estimating.

In [3]:
tickets["n_chars"] = tickets["ticket_text"].str.len()
tickets["n_words"] = tickets["ticket_text"].str.split().str.len()
tickets["n_tokens"] = tickets["ticket_text"].apply(count_tokens)

summary = tickets[["n_chars", "n_words", "n_tokens"]].agg(["mean", "min", "max"]).round(1)
print(summary.to_string())

ratio = tickets["n_chars"].sum() / tickets["n_tokens"].sum()
print(f"\ncharacters per token across the corpus : {ratio:.2f}")
print(f"tokens per word across the corpus      : {tickets['n_tokens'].sum() / tickets['n_words'].sum():.2f}")

      n_chars  n_words  n_tokens
mean     93.6     15.8      27.7
min      55.0     10.0      15.0
max     141.0     22.0      47.0

characters per token across the corpus : 3.38
tokens per word across the corpus      : 1.75


**Interpretation.** About **3.4 characters per token** and **1.75 tokens per word** on this corpus. The widely
quoted "~4 characters per token" rule of thumb lands close, which is a useful sanity check — but note it is a
*corpus-level average*. Any single ticket can sit well off it, so never budget a single request from the
average alone.

---

## Part 2 — The context window is a budget

The **context window** is the maximum number of tokens a model can hold at once — your system prompt, your
user prompt, the data you paste in, *and* the answer it generates, all together.

It behaves like a fixed bag: anything that does not fit does not get considered. So the practical question is
always "how much of my data actually fits, once the instructions and the reply have taken their share?"


In [4]:
def context_budget(window, prompt_overhead, reserve_for_output, per_item_tokens):
    """How many items fit in a context window after fixed costs are paid?"""
    usable = window - prompt_overhead - reserve_for_output
    return max(0, int(usable // per_item_tokens)), usable


avg_ticket = tickets["n_tokens"].mean()
worst_ticket = tickets["n_tokens"].max()
OVERHEAD, RESERVE = 220, 800          # instructions, and room for the model's answer

rows = []
for name, window in [("small (8K)", 8_000), ("medium (32K)", 32_000), ("large (128K)", 128_000)]:
    fit_avg, usable = context_budget(window, OVERHEAD, RESERVE, avg_ticket)
    fit_worst, _ = context_budget(window, OVERHEAD, RESERVE, worst_ticket)
    rows.append({"window": name, "usable_tokens": usable,
                 "tickets_if_average": fit_avg, "tickets_if_all_worst_case": fit_worst})

budget = pd.DataFrame(rows)
print(f"average ticket = {avg_ticket:.1f} tokens | longest ticket = {worst_ticket} tokens\n")
budget

average ticket = 27.7 tokens | longest ticket = 47 tokens



,window,usable_tokens,tickets_if_average,tickets_if_all_worst_case
0,small (8K),6980,252,148
1,medium (32K),30980,1119,659
2,large (128K),126980,4588,2701


**Interpretation.** Plan with the worst case, not the average. Sizing a batch on the average ticket and then
meeting a run of long ones is how a job that "worked in testing" starts silently dropping the tail of its input
in production. The gap between the two right-hand columns is the size of that risk.

In [5]:
# All 40 tickets at once - does the whole corpus fit in a small window?
total = tickets["n_tokens"].sum()
need = total + OVERHEAD + RESERVE
print(f"all {len(tickets)} tickets            : {total} tokens")
print(f"plus overhead and reserve  : {need} tokens")
print(f"fits in an 8K window?      : {need <= 8000}")

all 40 tickets            : 1107 tokens
plus overhead and reserve  : 2127 tokens
fits in an 8K window?      : True


This whole ticket file is small enough to send in one request. That will not be true of the next dataset you
meet, and the moment it is not, you need retrieval instead of pasting — which is precisely what session 13.1
(RAG) exists to solve.

---

## Part 3 — Probabilistic text generation

An LLM does not look up an answer. At each step it produces a **probability distribution over the next token**
and samples from it. Everything else — creativity, repetitiveness, hallucination — follows from that one fact.

To see it honestly, we will build a tiny **bigram model** from the real ticket text: for each token, count what
actually followed it in this corpus. It is far simpler than an LLM, but it is genuinely probabilistic and
genuinely trained on our data, so the behaviour it shows is real rather than staged.

For this section only we work at **word level** rather than the sub-word units of Part 1, purely so the
generated text is readable. The mechanism being demonstrated is identical either way.

In [6]:
def words(text):
    """Word-level split. Used only for the generation demo, so the output stays readable."""
    return [w.lower() for w in WORD_RE.findall(str(text))]


corpus_tokens = []
for t in tickets["ticket_text"]:
    corpus_tokens.extend(words(t))

bigrams = defaultdict(Counter)
for a, b in zip(corpus_tokens, corpus_tokens[1:]):
    bigrams[a][b] += 1

print(f"tokens in corpus       : {len(corpus_tokens)}")
print(f"distinct tokens        : {len(set(corpus_tokens))}")
print(f"contexts with a follower: {len(bigrams)}")

print("\nWhat actually follows 'order' in this corpus:")
for tok, n in bigrams["order"].most_common(6):
    print(f"   {tok:<12} {n}")

tokens in corpus       : 745
distinct tokens        : 339
contexts with a follower: 339

What actually follows 'order' in this corpus:
   mrd          5
   history      2
   ?            1
   .            1
   that         1
   or           1


That short table *is* a next-token distribution, learned from data. A real LLM does the same thing over a far
larger vocabulary with far more context — but the shape of the object is identical.

### Temperature

**Temperature** reshapes that distribution before sampling. Low temperature sharpens it toward the most likely
token (repeatable, conservative). High temperature flattens it (varied, riskier). Let us apply it to the real
counts above.

In [7]:
def apply_temperature(counter, temperature):
    """Convert raw counts into a sampling distribution at the given temperature."""
    toks = list(counter)
    counts = np.array([counter[t] for t in toks], dtype=float)
    if temperature <= 0:                       # greedy: always the top token
        probs = np.zeros_like(counts)
        probs[counts.argmax()] = 1.0
        return toks, probs
    logits = np.log(counts) / temperature
    exp = np.exp(logits - logits.max())
    return toks, exp / exp.sum()


for temp in [0.2, 1.0, 2.0]:
    toks, probs = apply_temperature(bigrams["order"], temp)
    top = sorted(zip(toks, probs), key=lambda x: -x[1])[:4]
    shown = "  ".join(f"{t}={p:.2f}" for t, p in top)
    print(f"temperature {temp:<4} -> {shown}")

temperature 0.2  -> mrd=0.99  history=0.01  ?=0.00  .=0.00
temperature 1.0  -> mrd=0.36  history=0.14  ?=0.07  .=0.07
temperature 2.0  -> mrd=0.21  history=0.13  ?=0.09  .=0.09


**Interpretation.** At temperature 0.2 the top token takes almost all the probability mass — the model will
say the same thing every time. At 2.0 the mass spreads out and unlikely continuations become live options. Same
model, same data; only the sampling rule changed.

In [8]:
def generate(seed_token, n=14, temperature=1.0, rng=rng):
    out, cur = [seed_token], seed_token
    for _ in range(n):
        if cur not in bigrams:
            break
        toks, probs = apply_temperature(bigrams[cur], temperature)
        cur = rng.choice(toks, p=probs)
        out.append(cur)
    return " ".join(out)


for temp in [0.2, 1.0, 2.0]:
    print(f"[temp {temp}] {generate('order', temperature=temp)}")
    print()

[temp 0.2] order mrd - 89550 . i have to my account after approval ? the warranty

[temp 1.0] order . k @ example . this needs immediate attention . how do i want

[temp 2.0] order mrd - 89104 . refund of use two debits of inr 800 short of



---

## Part 4 — Why hallucinations happen

Read the sentences above again. They are grammatical, they use the right vocabulary, they *sound* like support
tickets — and they state things that were never true of any real ticket.

Nothing malfunctioned. The model was asked for likely next tokens and gave likely next tokens. It was never
asked whether the result was true, and it has no mechanism for checking.

> **A hallucination is not a bug report.** It is the system working exactly as designed, applied to a question
> that needs a fact. Fluency is evidence of a good language model; it is not evidence of a true statement.

The practical consequence: you reduce hallucination by changing the *inputs and the checks*, not by asking the
model to try harder.

In [9]:
# The corpus never contains this claim - but the generator will happily build a fluent sentence around it.
print("Does any real ticket mention a 'lifetime warranty'? ",
      tickets["ticket_text"].str.contains("lifetime", case=False).any())
print("\nGenerated text starting from 'warranty':")
print("  ", generate("warranty", n=12, temperature=1.4))

Does any real ticket mention a 'lifetime warranty'?  False

Generated text starting from 'warranty':
   warranty but the website ? applied a coupon codes on 28 july for


**Interpretation.** The generator produces confident text around a token whose real support in the data is
thin. That is the mechanism behind hallucination in miniature: plausible continuation with no notion of truth.
Three defences follow directly, and all three appear later in this module — ground the model in retrieved
source text (13.1 RAG), constrain the output shape so it can be checked (session 30), and keep a human on
anything consequential (15.2).

---

## Part 5 — Prompt anatomy: system vs user

A prompt is not one blob of text. Two parts do different jobs:

- The **system prompt** sets standing behaviour — role, rules, output format, refusals. It applies to the whole
  conversation.
- The **user prompt** carries the specific request and the data for *this* turn.

Putting standing rules in the user prompt is the most common beginner mistake, because it means repeating them
(and paying for them) on every single turn, with no guarantee they outrank the user's own words.

In [10]:
CATEGORIES = sorted(tickets["category"].unique())

SYSTEM_PROMPT = (
    "You are a support ticket triage assistant for an Indian e-commerce company.\n"
    f"Classify each ticket into exactly one category: {', '.join(CATEGORIES)}.\n"
    "Rules:\n"
    "- Reply with the category name only, in lowercase. No explanation.\n"
    "- If the ticket does not clearly fit, reply: unclear\n"
    "- Never invent an order ID, refund amount, or delivery date."
)

print(SYSTEM_PROMPT)
print(f"\nsystem prompt cost: {count_tokens(SYSTEM_PROMPT)} tokens (paid once per request)")

You are a support ticket triage assistant for an Indian e-commerce company.
Classify each ticket into exactly one category: account, billing, delivery, how_to, product_issue, refund.
Rules:
- Reply with the category name only, in lowercase. No explanation.
- If the ticket does not clearly fit, reply: unclear
- Never invent an order ID, refund amount, or delivery date.

system prompt cost: 114 tokens (paid once per request)


### Zero-shot, few-shot and chain-of-thought

Three ways to build the user half of the prompt. We will construct each one properly and measure what it costs,
because token cost is the part students consistently underestimate.

In [11]:
sample = tickets.iloc[1]["ticket_text"]      # a delivery ticket


def zero_shot(ticket):
    return f"Ticket: {ticket}\nCategory:"


def few_shot(ticket, examples):
    lines = [f"Ticket: {r.ticket_text}\nCategory: {r.category}\n" for r in examples.itertuples()]
    return "\n".join(lines) + f"\nTicket: {ticket}\nCategory:"


def chain_of_thought(ticket):
    return (
        f"Ticket: {ticket}\n"
        "Think step by step:\n"
        "1. What is the customer asking for?\n"
        "2. Which category does that request belong to?\n"
        "3. State the final category on the last line as 'Category: <name>'."
    )


examples = tickets.groupby("category", group_keys=False).head(1)     # one example per category
prompts = {
    "zero_shot": zero_shot(sample),
    "few_shot": few_shot(sample, examples),
    "chain_of_thought": chain_of_thought(sample),
}

cost = pd.DataFrame(
    [{"strategy": k,
      "user_tokens": count_tokens(v),
      "total_with_system": count_tokens(SYSTEM_PROMPT) + count_tokens(v)}
     for k, v in prompts.items()]
)
cost

,strategy,user_tokens,total_with_system
0,zero_shot,31,145
1,few_shot,279,393
2,chain_of_thought,82,196


In [12]:
print(prompts["few_shot"][:400], "...\n")
print("-" * 70)
print(prompts["chain_of_thought"])

Ticket: Ordered a wireless keyboard on 28 July, delivered cracked. I want a refund not a replacement. Order MRD-88213. Reach me at arun.k@example.com
Category: refund

Ticket: Where is my order? It says shipped since Friday but tracking has not moved at all.
Category: delivery

Ticket: I was charged twice for the same order. Two debits of INR 4599 on my card. Please reverse one immediately. Call 9 ...

----------------------------------------------------------------------
Ticket: Where is my order? It says shipped since Friday but tracking has not moved at all.
Think step by step:
1. What is the customer asking for?
2. Which category does that request belong to?
3. State the final category on the last line as 'Category: <name>'.


**Interpretation.** Few-shot is by far the most expensive of the three here — roughly nine times the user-prompt
cost of zero-shot — because every example is re-sent on every single request. At scale that is the dominant
line on your bill, and it buys nothing on tickets the model already handles correctly. Chain-of-thought is
cheap to *send* but expensive to *receive*, since the reasoning comes back in the output tokens.

---

## Part 6 — Does more context actually help?

A fair question: if examples cost that much, do they earn it?

We cannot answer that for a real LLM offline. What we *can* do is run a controlled experiment on a keyword
matcher and see what it says about **context quality versus context volume**.

> **Read this for what it is.** The matcher below is TF-IDF cosine similarity, not a language model. It does not
> do in-context learning the way an LLM does, so this is *not* evidence about how few-shot prompting performs
> on GPT or Gemini. It is a clean demonstration of one narrower point: what you put in the context matters more
> than how much you put in.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

CATEGORY_DESCRIPTIONS = {
    "refund":        "refund money back return credited reversal",
    "delivery":      "delivery shipped courier parcel tracking address dispatch",
    "billing":       "billing invoice charged payment tax gst emi coupon amount",
    "product_issue": "product defective damaged broken wrong item quality fault",
    "how_to":        "how do i question policy warranty process guide can i",
    "account":       "account login password profile id merge delete data",
}


def score_reference(reference, train, test):
    vec = TfidfVectorizer(stop_words="english")
    ref_texts = [reference[c] for c in CATEGORIES]
    vec.fit(list(test["ticket_text"]) + ref_texts)
    sims = cosine_similarity(vec.transform(test["ticket_text"]), vec.transform(ref_texts))
    pred = [CATEGORIES[i] for i in sims.argmax(axis=1)]
    return (np.array(pred) == test["category"].values).mean()


records = []
for seed in [0, 1, 42, 7, 13]:
    train, test = train_test_split(tickets, test_size=0.5,
                                   stratify=tickets["category"], random_state=seed)
    curated = score_reference(CATEGORY_DESCRIPTIONS, train, test)
    raw = {c: " ".join(train[train["category"] == c]["ticket_text"].head(3)) for c in CATEGORIES}
    verbose = score_reference(raw, train, test)
    combined = score_reference({c: CATEGORY_DESCRIPTIONS[c] + " " + raw[c] for c in CATEGORIES}, train, test)
    records.append({"seed": seed, "curated_keywords": curated,
                    "raw_examples": verbose, "both": combined})

experiment = pd.DataFrame(records)
print(experiment.round(3).to_string(index=False))
print("\nmeans:")
print(experiment[["curated_keywords", "raw_examples", "both"]].mean().round(3).to_string())

 seed  curated_keywords  raw_examples  both
    0              0.65          0.40  0.50
    1              0.70          0.50  0.60
   42              0.70          0.35  0.55
    7              0.70          0.55  0.70
   13              0.70          0.50  0.55

means:
curated_keywords    0.69
raw_examples        0.46
both                0.58


**Interpretation — and it is not the expected result.** The curated keyword description wins on **all five
splits**, averaging **0.69** against **0.46** for three raw ticket examples per category. Adding the examples to
the curated text (`both`, 0.58) makes things *worse* than the curated text alone.

The reason is signal dilution: a real ticket is mostly words that carry no category information — greetings,
order numbers, dates, apologies — and in a bag-of-words matcher that noise drowns the few words that matter.

The transferable lesson, stated carefully: **a longer context is not automatically a better context.** Before
you pay to send more, check that what you are sending carries signal. For a real LLM you would run this same
comparison against the actual model rather than assuming either way — which is exactly the habit session 30
sets up.

---

## Exercises

Try each one before opening the solution.

### Exercise 1 — The most expensive ticket

Which ticket costs the most tokens to send, and how many? Compare it to the corpus average.

<details><summary>Solution</summary>

```python
worst = tickets.loc[tickets["n_tokens"].idxmax()]
print(worst["ticket_id"], worst["n_tokens"], "tokens")
print(f"{worst['n_tokens'] / tickets['n_tokens'].mean():.2f}x the average")
```

**TKT-1001 at 47 tokens**, about **1.70×** the 27.7-token average. Budgeting a batch on the average would
under-provision by nearly half on this ticket alone.

</details>

In [14]:
# TODO - Exercise 1


### Exercise 2 — Greedy decoding

Generate from the seed token `order` at temperature `0` three times. Do you get the same text each time, and
why?

<details><summary>Solution</summary>

```python
for _ in range(3):
    print(generate("order", n=10, temperature=0))
```

All three runs are **identical**, because temperature 0 puts all probability on the single most likely token —
there is nothing left to sample. This is why temperature 0 is the right setting for classification or
extraction, where you want the same input to give the same answer every time.

</details>

In [15]:
# TODO - Exercise 2


### Exercise 3 — Cost of few-shot at scale

Your system classifies 5,000 tickets a day. Using the token counts computed in Part 5, how many *extra* tokens
per day does few-shot cost compared with zero-shot?

<details><summary>Solution</summary>

```python
zs = cost.loc[cost["strategy"] == "zero_shot", "total_with_system"].iloc[0]
fs = cost.loc[cost["strategy"] == "few_shot", "total_with_system"].iloc[0]
print(f"per request: {fs - zs} extra tokens")
print(f"per day    : {(fs - zs) * 5000:,} extra tokens")
```

**248 extra tokens per request**, so **1,240,000 extra tokens per day** — over a million tokens daily, bought
before a single ticket is answered. Worth it only if you can show the accuracy it buys.

</details>

In [16]:
# TODO - Exercise 3


---

## Recap, mapped to the learning objectives

**LO 1 — Tokens and the context window.** Text is cut into sub-word tokens (~3.6 characters each on this
corpus). The context window is a hard budget shared by your instructions, your data and the model's reply, and
it must be planned against the worst-case item, not the average.

**LO 2 — Probabilistic generation and hallucination.** A bigram model built from the real tickets showed a
genuine next-token distribution, temperature reshaping it, and fluent output that states things no ticket ever
said. Hallucination is that mechanism working normally, not a defect to be scolded out of the model.

**LO 3 — Prompt strategies.** System prompts hold standing rules; user prompts carry the turn. Zero-shot,
few-shot and chain-of-thought were built and priced — few-shot cost ~6× the user tokens of zero-shot, and a
controlled experiment showed that padding a context with verbose examples can actively hurt.

The sentence to carry into session 30: **you control an LLM through what you put in its context and what you
check on the way out — everything else is hope.**